# Image Data Processing

This notebook loads images from **10 class folders** located in the same directory
as this notebook (your VSCode workspace / repo root) and normalizes every pixel
value to the range **[0, 1]** using:

```python
image = image / 255.0
```

## Expected folder layout

```
repo-root/               ← base_dir (default: this notebook's directory)
├── class_0/
│   ├── img1.jpg
│   └── ...
├── class_1/
│   └── ...
...
└── class_9/
    └── ...
```

If your folders live somewhere else, change `BASE_DIR` in the **Configuration** cell below.

In [ ]:
import os
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# BASE_DIR defaults to the directory that contains this notebook, which is
# the repo root in your VSCode workspace.  Change it if your image folders
# are located somewhere else.
# ---------------------------------------------------------------------------
BASE_DIR = Path(
    globals().get('__vsc_ipynb_file__')  # VSCode sets this automatically
    or (__file__ if '__file__' in dir() else '.')  # fallback for other environments
).parent

# Supported image file extensions
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

print(f'BASE_DIR: {BASE_DIR.resolve()}')

In [ ]:
def load_and_normalize_images(folder_path: Path):
    """
    Load all images from *folder_path* and normalize pixel values to [0, 1].

    Supports:
    - 8-bit  (uint8)   →  divided by 255.0
    - 16-bit (uint16)  →  divided by 65535.0
    - float            →  clipped to [0, 1]

    Parameters
    ----------
    folder_path : Path
        Directory containing image files.

    Returns
    -------
    list of numpy.ndarray
        Normalized float64 image arrays.
    """
    folder_path = Path(folder_path)
    if not folder_path.is_dir():
        print(f'Warning: "{folder_path}" is not a valid directory. Skipping.')
        return []

    images = []
    for img_path in sorted(folder_path.iterdir()):
        if img_path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(img_path) as img:
                    image = np.array(img)
                # Normalize to [0, 1] based on dtype
                if image.dtype == np.uint8:
                    image = image / 255.0
                elif image.dtype == np.uint16:
                    image = image / 65535.0
                elif np.issubdtype(image.dtype, np.floating):
                    image = np.clip(image, 0.0, 1.0)
                else:
                    image = image / 255.0
                images.append(image)
            except Exception as e:
                print(f'Warning: could not load "{img_path}": {e}')
    return images

In [ ]:
# Load and normalize images from all 10 class folders
all_images = {}  # folder_name -> list of normalized arrays

if not BASE_DIR.is_dir():
    print(f'BASE_DIR "{BASE_DIR}" does not exist. Please check the path.')
else:
    folders = sorted(
        [p for p in BASE_DIR.iterdir() if p.is_dir()]
    )[:10]  # at most 10 folders

    print(f'Found {len(folders)} folder(s):')
    for folder in folders:
        normalized = load_and_normalize_images(folder)
        all_images[folder.name] = normalized
        print(f'  {folder.name}: {len(normalized)} image(s) loaded')

In [ ]:
# Display one sample image per folder (first image, if available)
n_folders = len(all_images)
if n_folders == 0:
    print('No folders found. Please check BASE_DIR is set correctly.')
else:
    cols = min(5, n_folders)
    rows = (n_folders + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax, (folder, images) in zip(axes, all_images.items()):
        if images:
            ax.imshow(images[0], cmap='gray' if images[0].ndim == 2 else None)
            ax.set_title(f'{folder}\n(normalized)', fontsize=9)
        else:
            ax.set_title(f'{folder}\n(no images)', fontsize=9)
        ax.axis('off')
    # Hide any unused axes
    for ax in axes[n_folders:]:
        ax.set_visible(False)
    plt.suptitle('Sample Normalized Images (pixel values in [0, 1])', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Verify normalization: pixel values should be in [0, 1]
for folder, images in all_images.items():
    for img in images:
        assert img.min() >= 0.0, f'Min value below 0 in folder {folder}'
        assert img.max() <= 1.0, f'Max value above 1 in folder {folder}'

print('All images are correctly normalized to [0, 1].')